# GenMusic full-dataset training on Google Colab

Backend này chạy song song với Kaggle, không thay thế các launcher Kaggle. Notebook tải sáu output preprocessing public, ghép đúng 1.843 record, train 40 epoch, cache từng phần dữ liệu và lưu checkpoint giữa epoch vào Google Drive để tiếp tục sau khi runtime bị ngắt.

Trước khi chạy: chọn **Runtime → Change runtime type → T4 GPU**. Để tránh paste token mỗi lần, có thể thêm Colab secret `KAGGLE_API_TOKEN` và bật quyền truy cập notebook.

In [ ]:
# Colab backend configuration. Kaggle launchers are unchanged.
COLAB_NOTEBOOK_URL = 'https://colab.research.google.com/drive/1zGT80eSQdyUjP6rMxY0WWsAo-xEVd8GD?usp=sharing'
REPO_URL = 'https://github.com/nhuy5602/GenMusic.git'
REPO_REF = 'master'
DRIVE_ROOT = "/content/drive/MyDrive/GenMusic"
WORKSPACE = "/content/genmusic_colab"
EXPECTED_RECORDS = 1843
EPOCHS = 40
BATCH_SIZE = 4
CACHE_DATA_ON_DRIVE = True
CHECKPOINT_EVERY_STEPS = 25
SKIP_PREFLIGHT_TRAIN = True
KERNEL_REFS = ['ngochuy5602/genmusic-prep-p1-1784095999', 'ngochuy5602/genmusic-prep-p2-1784096002', 'ngochuy5602/genmusic-prep-p3-1784107963', 'ngochuy5602/genmusic-prep-p4-1784108049', 'ngochuy5602/genmusic-prep-p5-1784120546', 'ngochuy5602/genmusic-fullexp-1784078352']


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import shutil
import subprocess

if shutil.which("nvidia-smi") is None:
    raise RuntimeError(
        "Chưa có GPU. Chọn Runtime > Change runtime type > T4 GPU rồi chạy lại."
    )
subprocess.run(["nvidia-smi", "-L"], check=True)


In [ ]:
import os
import pathlib
import subprocess
import sys

repo_root = pathlib.Path("/content/GenMusic")
if (repo_root / ".git").is_dir():
    subprocess.run(["git", "-C", str(repo_root), "fetch", "origin", REPO_REF], check=True)
    subprocess.run(["git", "-C", str(repo_root), "checkout", REPO_REF], check=True)
    subprocess.run(["git", "-C", str(repo_root), "pull", "--ff-only", "origin", REPO_REF], check=True)
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(repo_root)],
        check=True,
    )

subprocess.run(
    ["apt-get", "update", "-qq"],
    check=True,
)
subprocess.run(
    ["apt-get", "install", "-y", "-qq", "espeak-ng", "ffmpeg"],
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        "kaggle>=2.0.0",
        "transformers==5.13.1",
        "sentencepiece",
        "librosa",
        "soundfile",
        "vocos",
        "imageio-ffmpeg",
        "matplotlib",
    ],
    check=True,
)
os.environ["PYTHONPATH"] = str(repo_root) + os.pathsep + os.environ.get("PYTHONPATH", "")
print("Source and dependencies are ready:", repo_root)


In [ ]:
import getpass
import os

try:
    from google.colab import userdata
    kaggle_token = userdata.get("KAGGLE_API_TOKEN")
except Exception:
    kaggle_token = ""

if not kaggle_token:
    kaggle_token = getpass.getpass(
        "Paste KAGGLE_API_TOKEN (KGAT_...). Token is kept only in this runtime: "
    ).strip()
if not kaggle_token.startswith(("KGAT_", "KGAT-")):
    raise ValueError("KAGGLE_API_TOKEN must start with KGAT_ or KGAT-")
os.environ["KAGGLE_API_TOKEN"] = kaggle_token
print("Kaggle token loaded into the current Colab runtime.")


In [ ]:
command = [
    sys.executable,
    "-u",
    str(repo_root / "scripts" / "run_colab_full_training.py"),
    "--workspace",
    WORKSPACE,
    "--drive-root",
    DRIVE_ROOT,
    "--expected-records",
    str(EXPECTED_RECORDS),
    "--epochs",
    str(EPOCHS),
    "--batch-size",
    str(BATCH_SIZE),
    "--checkpoint-every-steps",
    str(CHECKPOINT_EVERY_STEPS),
]
if CACHE_DATA_ON_DRIVE:
    command.append("--cache-data-on-drive")
if SKIP_PREFLIGHT_TRAIN:
    command.append("--skip-preflight-train")
for kernel_ref in KERNEL_REFS:
    command.extend(["--kernel", kernel_ref])

print("Starting full Colab pipeline...")
subprocess.run(command, cwd=repo_root, check=True)


In [ ]:
from IPython.display import Audio, display
from pathlib import Path

result_dir = Path(DRIVE_ROOT) / "generated_all_parts"
mp3_files = sorted(result_dir.glob("*.mp3"))
wav_files = sorted(result_dir.glob("*.wav"))
print("Checkpoint:", Path(DRIVE_ROOT) / "checkpoints" / "baseline_all_parts.pt")
print("State:", Path(DRIVE_ROOT) / "colab_training_state.json")
print("Generated files:", [str(path) for path in mp3_files + wav_files])
if mp3_files:
    display(Audio(str(mp3_files[0])))
elif wav_files:
    display(Audio(str(wav_files[0])))
